# Gemini finetune test

In [3]:
# Read parallel sentences file
import os
import json

filename = "generated_sentences/kalamang_generated_sentences.json"

if os.path.exists(filename):
    with open(filename, "r", encoding="utf-8") as file:
        data = json.load(file)
        for entry in data:
            print(f"Source: {entry['generated_text']}")
            print(f"Target: {entry['generated_text_translation']}")
    print(f"Loaded {len(data)} entries from {filename}")

else:
    print(f"File {filename} does not exist.")

Source: bal se sor=at koraru sou iam fish=obj bite
Target: ‘The dog has bitten the slide.’
Source: bal se sor=at koraru owa iam fish=obj bite
Target: ‘The dog has bitten the over there.’
Source: bal se sor=at koraru nahimat iam fish=obj bite
Target: ‘The dog has bitten the thrifty.’
Source: bal se sor=at koraru osatko iam fish=obj bite
Target: ‘The dog has bitten the dem up there.’
Source: bal se sor=at koraru run iam kiem=obj bite
Target: The dog has bitten the run.
Source: bal se sor=at koraru nalat iam fish=obj bite
Target: ‘The dog has bitten the rat.’
Source: bal se sor=at koraru mukmuk iam rock=obj bite
Target: ‘The dog has bitten the rock.’
Source: bal se sor=at koraru kolo iam fish=obj take out
Target: ‘The dog has taken out the fish.’
Source: bal se sor=at koraru osie iam rest=obj bite
Target: The dog has bitten the rest.
Source: bal se sor=at koraru wise iam fish=obj bite
Target: The wise has bitten the fish.
Source: ma reitkon purap-i an=at slide=et
Target: ‘He sent me the s

In [7]:
# Clean sentences (Keep all entries within quotes ‘’), if not, keep the whole
cleaned_data = []
for entry in data:
    original = entry['generated_text']
    translated = entry['generated_text_translation']

    # Extract text within quotes if present
    if '‘' in original and '’' in original:
        start = original.index('‘') + 1
        end = original.index('’')
        original = original[start:end].strip()
    
    if '‘' in translated and '’' in translated:
        start = translated.index('‘') + 1
        end = translated.index('’')
        translated = translated[start:end].strip()

    # If "Translation: " prefix exists, remove it
    if translated.lower().startswith("translation:"):
        translated = translated[len("translation:"):].strip()
    
    if translated.lower().startswith("translation -"):
        translated = translated[len("translation -"):].strip()
    if translated.lower().startswith("translation in english:"):
        translated = translated[len("translation in english:"):].strip()

    # Do it for all other types of quotes as well
    quote_pairs = [('“', '”'), ('‘', '’'), ('"', '"'), ("'", "'")]
    for open_quote, close_quote in quote_pairs:
        if translated.startswith(open_quote) and translated.endswith(close_quote):
            translated = translated[1:-1].strip()
        if original.startswith(open_quote) and original.endswith(close_quote):
            original = original[1:-1].strip()
    cleaned_data.append({
        'original_sentence': original,
        'translated_sentence': translated
    })

cleaned_data

[{'original_sentence': 'bal se sor=at koraru sou iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the slide.'},
 {'original_sentence': 'bal se sor=at koraru owa iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the over there.'},
 {'original_sentence': 'bal se sor=at koraru nahimat iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the thrifty.'},
 {'original_sentence': 'bal se sor=at koraru osatko iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the dem up there.'},
 {'original_sentence': 'bal se sor=at koraru run iam kiem=obj bite',
  'translated_sentence': 'The dog has bitten the run.'},
 {'original_sentence': 'bal se sor=at koraru nalat iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the rat.'},
 {'original_sentence': 'bal se sor=at koraru mukmuk iam rock=obj bite',
  'translated_sentence': 'The dog has bitten the rock.'},
 {'original_sentence': 'bal se sor=at koraru kolo iam fish=obj take out',
  'transl

In [11]:
data = cleaned_data

In [14]:
import random
train_test_split = 0.8

# Randomly shuffle data
random.shuffle(data)

train_size = int(len(data) * train_test_split)
train_data = data[:train_size]
test_data = data[train_size:]

len(train_data), len(test_data)

(800, 200)

In [16]:
test_data

[{'original_sentence': 'kolo mun = ten wandi=et ka bisa na=ta',
  'translated_sentence': 'You can eat rotten things like this.'},
 {'original_sentence': 'an se nalat min=kin',
  'translated_sentence': 'I already wanted to die.'},
 {'original_sentence': 'an se koi yal yal yal tebol-suban war=i war=i eh sor nat=nin adv long ago - wise',
  'translated_sentence': 'I paddled and paddled again, fished at the reef edge, fished and fished, [narr8_1:02] the wise didn'},
 {'original_sentence': 'koni kararu koni kiem kon-i kararu kon-i kiem',
  'translated_sentence': '[You] lift one run, leave one.'},
 {'original_sentence': 'what',
  'translated_sentence': 'dem up there-3poss=obj'},
 {'original_sentence': '* nalat-gan temun',
  'translated_sentence': 'All trees are big.'},
 {'original_sentence': 'an seseri koːyet nahimat koyet (...) pasori koːyet an seser=i koyet nahimat=i koyet (...) pasor=i koyet',
  'translated_sentence': 'After I'},
 {'original_sentence': 'what', 'translated_sentence': 'wise-

In [ ]:
# Modify into gemini finetune format
output_filename = "train.jsonl"
output_path = "gemini_finetune_data"

target_language = "kalamang"

combined_output_path = os.path.join(output_path, target_language)
os.makedirs(combined_output_path, exist_ok=True)
output_filename = os.path.join(combined_output_path, output_filename)


with open(output_filename, "w", encoding="utf-8") as outfile:
    for entry in train_data:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {target_language}: " + entry["original_sentence"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translated_sentence"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saved {len(train_data)} training entries to {output_filename}")

output_filename = os.path.join(combined_output_path, "test.jsonl")
with open(output_filename, "w", encoding="utf-8") as outfile:
    for entry in test_data:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {target_language}: " + entry["original_sentence"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translated_sentence"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saved {len(test_data)} testing entries to {output_filename}")


Saved 800 training entries to gemini_finetune_data\kalamang\train.jsonl
{'original_sentence': 'kolo mun = ten wandi=et ka bisa na=ta', 'translated_sentence': 'You can eat rotten things like this.'}
{'original_sentence': 'an se nalat min=kin', 'translated_sentence': 'I already wanted to die.'}
{'original_sentence': 'an se koi yal yal yal tebol-suban war=i war=i eh sor nat=nin adv long ago - wise', 'translated_sentence': 'I paddled and paddled again, fished at the reef edge, fished and fished, [narr8_1:02] the wise didn'}
{'original_sentence': 'koni kararu koni kiem kon-i kararu kon-i kiem', 'translated_sentence': '[You] lift one run, leave one.'}
{'original_sentence': 'what', 'translated_sentence': 'dem up there-3poss=obj'}
{'original_sentence': '* nalat-gan temun', 'translated_sentence': 'All trees are big.'}
{'original_sentence': 'an seseri koːyet nahimat koyet (...) pasori koːyet an seser=i koyet nahimat=i koyet (...) pasor=i koyet', 'translated_sentence': 'After I'}
{'original_sente